# Lab 07 Challenge: Build a Mini Agent

**Goal:** Combine reasoning + tools + memory into a working agent.

Your mission: Build a "Smart Study Buddy" agent that:
1. Uses Chain-of-Thought to reason about questions
2. Has access to tools (calculator, dictionary, quiz generator)
3. Remembers the conversation (short-term memory)
4. Uses the ReAct pattern to decide when to use tools

This exercise has minimal pre-written code — use what you learned in Labs 01-06 to build it!

## Setup: Imports and LLM

In [ ]:
import json
import math
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOllama(model="llama3.2:1b")

## Part A: Define Your Tools

Here are some starter tools. You can add more!

In [ ]:
def calculator(expression: str) -> str:
    """Calculate a math expression."""
    try:
        allowed = {"__builtins__": {}, "math": math}
        return str(eval(expression, allowed))
    except Exception as e:
        return f"Error: {e}"


def define_word(word: str) -> str:
    """Look up the definition of a word."""
    definitions = {
        "algorithm": "A step-by-step procedure for solving a problem or accomplishing a task.",
        "api": "Application Programming Interface \u2014 a way for software programs to communicate with each other.",
        "variable": "A named storage location in a program that holds a value which can change.",
        "function": "A reusable block of code that performs a specific task.",
        "loop": "A programming construct that repeats a block of code multiple times.",
        "recursion": "When a function calls itself to solve smaller instances of the same problem.",
        "testing": "The practice of verifying that software behaves as expected through automated or manual checks.",
        "database": "An organized collection of structured data stored electronically for efficient retrieval and management.",
    }
    return definitions.get(word.lower().strip(), f"Definition not found for: {word}")


def generate_quiz(topic: str) -> str:
    """Generate a quick quiz question about a topic."""
    quizzes = {
        "python": "Q: What keyword is used to define a function in Python?\nA) func  B) def  C) function  D) define\nCorrect: B) def",
        "git": "Q: What command creates a new Git branch?\nA) git new  B) git create  C) git branch  D) git fork\nCorrect: C) git branch",
        "testing": "Q: Which testing framework is commonly used in Python?\nA) JUnit  B) pytest  C) Mocha  D) RSpec\nCorrect: B) pytest",
        "linux": "Q: What command lists files in a directory?\nA) dir  B) show  C) ls  D) list\nCorrect: C) ls",
    }
    for key, quiz in quizzes.items():
        if key in topic.lower():
            return quiz
    return f"No quiz available for: {topic}. Try: python, git, testing, linux."

In [ ]:
TOOLS = {
    "calculator": {"fn": calculator, "desc": "Calculate a math expression (e.g., '17 * 28', 'math.sqrt(144)')"},
    "define_word": {"fn": define_word, "desc": "Look up the definition of a programming/tech term"},
    "generate_quiz": {"fn": generate_quiz, "desc": "Generate a quiz question about a topic (python, git, testing, linux)"},
}

## Part B: Build the System Prompt

**TODO 1:** Create a system prompt that tells the LLM:
1. Its role (Smart Study Buddy for programming students)
2. Available tools — list each tool with its name and what it does (read from the TOOLS dict)
3. The JSON format for tool calls: `{"tool": "tool_name", "argument": "the argument"}`
4. Instructions to answer directly when no tool is needed
5. Encouragement to think step by step for complex questions

**Hint:** Build the tool list dynamically from the TOOLS dict so adding new tools automatically updates the prompt.

In [ ]:
# TODO 1: Build SYSTEM_PROMPT
# Hint: Loop through TOOLS to build the tool descriptions dynamically
SYSTEM_PROMPT = "___"

## Part C: Build the Agent Loop

**TODO 2:** Implement `run_agent(user_message, history)` — the core ReAct loop.

The function should:
- Take a user message and conversation history; return `(response_text, updated_history)`
- Decide each turn: does the LLM want to call a tool, or answer directly?
- Execute the tool if one is requested and feed the result back to the LLM
- Cap tool calls at 3 per turn to prevent infinite loops

**Hint:** Use `response_text.index("{")` and `response_text.rindex("}")` to extract JSON from the LLM response.

In [ ]:
def run_agent(user_message: str, history: list) -> tuple[str, list]:
    """
    Run one turn of the agent.

    Args:
        user_message: The user's input
        history: List of previous messages (memory)

    Returns:
        (agent_reply, updated_history)
    """
    # TODO 2: Implement the ReAct loop
    # Think: how does the agent decide to call a tool vs. answer directly?
    # How should it update memory after each tool call?
    # Max 3 tool calls per turn. Return (response_text, updated_history).
    return "___", history

## Part D: Test and Extend

First, run the test below to see your agent in action. Then add a new tool!

**TODO 3:** Add an `example_code` tool that takes a programming topic (e.g., `"loop"`, `"class"`, `"function"`) and returns a **short, runnable code snippet** with a comment explaining what it does. Steps:
1. Define the `example_code` function — the dict values should be actual code strings (not just text definitions)
2. Add it to the TOOLS dict
3. Rebuild SYSTEM_PROMPT (call your prompt builder again)
4. Test with: `"Can you show me a code example of a loop?"`

In [ ]:
# Test the agent
history = [SystemMessage(content=SYSTEM_PROMPT)]

test_messages = [
    "Hi! I'm learning programming. Can you help me?",
    "What does 'algorithm' mean?",
    "What is 17 multiplied by 2847?",                            # Forces calculator use
    "Give me a quiz about Python!",
    "What were the two topics I asked you to look up or calculate?",  # Tests memory
]

for msg in test_messages:
    print(f"\nYou: {msg}")
    reply, history = run_agent(msg, history)
    print(f"Bot: {reply}")
    print(f"     [Memory: {len(history)} messages]")

In [ ]:
# TODO 3: Add explain_code tool, rebuild prompt, and test

In [ ]:
# Validation
score = 0
checks = []

# Check TODO 1: SYSTEM_PROMPT
if SYSTEM_PROMPT == "___":
    checks.append(("System prompt built", "TODO"))
    checks.append(("Prompt mentions tools", "TODO"))
    checks.append(("Prompt has JSON format", "TODO"))
else:
    if len(SYSTEM_PROMPT) > 50:
        checks.append(("System prompt built (length > 50)", "PASS"))
        score += 1
    else:
        checks.append(("System prompt built (too short)", "FAIL"))
    if "calculator" in SYSTEM_PROMPT and "define_word" in SYSTEM_PROMPT:
        checks.append(("Prompt mentions tools", "PASS"))
        score += 1
    else:
        checks.append(("Prompt mentions tools", "FAIL"))
    if '{"tool"' in SYSTEM_PROMPT or '"tool"' in SYSTEM_PROMPT:
        checks.append(("Prompt has JSON format instruction", "PASS"))
        score += 1
    else:
        checks.append(("Prompt has JSON format instruction", "FAIL"))

# Check TODO 2: run_agent — basic response + tool invocation
test_history = [SystemMessage(content=SYSTEM_PROMPT if SYSTEM_PROMPT != "___" else "test")]
result, _ = run_agent("Hello", test_history)
if result == "___":
    checks.append(("run_agent returns response", "TODO"))
    checks.append(("run_agent invokes tools", "TODO"))
else:
    if isinstance(result, str) and len(result) > 0:
        checks.append(("run_agent returns response", "PASS"))
        score += 1
    else:
        checks.append(("run_agent returns response", "FAIL"))
    # Verify tool invocation: a math query should grow history beyond 3 messages
    # (SystemMessage + HumanMessage + AIMessage(tool_call) + HumanMessage(result) + AIMessage(final) = 5)
    tool_history = [SystemMessage(content=SYSTEM_PROMPT if SYSTEM_PROMPT != "___" else "test")]
    _, tool_history = run_agent("What is 17 multiplied by 2847?", tool_history)
    if len(tool_history) >= 5:
        checks.append(("run_agent invokes tools (history grew)", "PASS"))
        score += 1
    else:
        checks.append(("run_agent invokes tools (answered without tool call)", "FAIL"))

# Check TODO 3: example_code tool
if "example_code" in TOOLS:
    checks.append(("example_code tool added", "PASS"))
    score += 1
    r = TOOLS["example_code"]["fn"]("loop")
    if isinstance(r, str) and "\n" in r:
        checks.append(("example_code returns a code snippet", "PASS"))
        score += 1
    else:
        checks.append(("example_code returns a code snippet (no code found)", "FAIL"))
else:
    checks.append(("example_code tool added", "TODO"))
    checks.append(("example_code returns a code snippet", "TODO"))

for check, status in checks:
    print(f"[{status}] {check}")
print(f"\nScore: {score}/7")

## Part E (Bonus): Make It Interactive

Uncomment the code below to run an interactive chat loop with the Study Buddy.
Type `quit` to exit.

In [ ]:
# Uncomment to run interactively:

# history = [SystemMessage(content=SYSTEM_PROMPT)]
# while True:
#     user_input = input("\nYou: ").strip()
#     if user_input.lower() == "quit":
#         print("Goodbye! Keep learning!")
#         break
#     reply, history = run_agent(user_input, history)
#     print(f"Bot: {reply}")

## Part F (Bonus): Add Even More Tools

Ideas for additional tools beyond `example_code`:
- `compare(a_and_b)`: Compare two technologies (e.g., "Python vs JavaScript")
- `acronym(letters)`: Expand a tech acronym (API, REST, SQL)
- `debug_hint(error)`: Return a tip for a common error type (NameError, IndexError, TypeError)

Add them to `TOOLS` and rebuild `SYSTEM_PROMPT` (your dynamic builder handles it automatically!).

## Takeaways

- You built a working agent from scratch combining **ReAct reasoning**, **tool use**, and **conversational memory**
- Dynamic prompt building means adding tools is effortless — just add to TOOLS and rebuild
- Memory lets the bot reference earlier messages (e.g., "What two topics did I ask about?")
- The ReAct loop decides whether to call a tool or answer directly — and feeds tool results back to the LLM
- Check `solutions/lab07_challenge.ipynb` for the complete version with all TODOs filled in